# Fin-JEPA — EUR/USD hourly pretraining on Colab GPU

This notebook runs `train_forex_h1.py` (latent-only Fin-JEPA, paper-faithful).

**Two ways to get the code + data here:**
- **(A) Clone from GitHub** (run the next code cell). Requires the repo to be pushed
  (one `git push` on your machine). Public repo works as-is; for private, use a token URL.
- **(B) Manual upload:** in Colab's left file pane, upload `model.py`, `alphas.py`,
  `forex_features.py`, `train_forex_h1.py`, and the folder `data/EURUSD_H1.csv`
  into this notebook's working directory, then **skip the clone cell**.

Then run the training cell (GPU). A few epochs is enough to confirm the pipeline.

In [ ]:
# (A) Clone from GitHub — only needed if you did NOT upload files manually.
# Edit the URL if the repo is private: https://<TOKEN>@github.com/jsnayem/fin-jepa.git
import os, subprocess, shutil
if not os.path.exists('fin-jepa'):
    !git clone https://github.com/jsnayem/fin-jepa.git fin-jepa
%cd fin-jepa
!ls -la


In [ ]:
# Install the only missing dependency (Colab already has torch[pip] + pandas + numpy)
!pip install -q einops
import torch
print('CUDA available:', torch.cuda.is_available(), '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')


In [ ]:
# Sanity check that all required files + data are present before training
import os
needed = ['model.py','alphas.py','forex_features.py','train_forex_h1.py','data/EURUSD_H1.csv']
missing = [f for f in needed if not os.path.exists(f)]
print('MISSING:', missing if missing else 'none — ready to train')


In [ ]:
# Train a few epochs on GPU. Increase --epochs later (scripted default 40).
!python train_forex_h1.py --epochs 3 --batch 256 --device cuda --ckpt checkpoints/forex_h1


In [ ]:
# Inspect the saved run metadata
import json
with open('checkpoints/forex_h1/meta.json') as f:
    print(json.dumps(json.load(f), indent=2))


In [ ]:
# Downstream probe + Value-of-Explanation on the validation split.
# Reports IC / rank-IC / R^2 / directional AUC of the probe head, and whether the
# JEPA latent error is informative about the forward mega-alpha move (VoE).
!python probe_forex_h1.py --ckpt checkpoints/forex_h1/best.pt --tau 24 --device cuda --out checkpoints/forex_h1/probe.json
with open('checkpoints/forex_h1/probe.json') as f:
    print(json.dumps(json.load(f), indent=2))
